In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [34]:
import os

files = os.listdir("/content")

for file in files:
    print(file)

.config
afl_match_features_v1.zip
__pycache__
afl_datasets.zip
predict.py
afl_player_features_v1.zip
afl_feature_dictionary_v1.csv
sample_data


In [39]:
import zipfile
import os

# Open the ZIP file
with zipfile.ZipFile("/content/afl_datasets.zip", "r") as zip_ref:
  zip_ref.extractall("/content")


# Extract match features
with zipfile.ZipFile("/content/afl_match_features_v1.zip", "r") as zip_ref:
    zip_ref.extractall("/content")

# Extract player features
with zipfile.ZipFile("/content/afl_player_features_v1.zip", "r") as zip_ref:
    zip_ref.extractall("/content")

print("Extraction completed.")

Extraction completed.


In [40]:
# Load Day 1 feature tables

match_features = pd.read_csv("afl_match_features_v1.csv")
player_features = pd.read_csv("afl_player_features_v1.csv")
feature_dictionary = pd.read_csv("afl_feature_dictionary_v1.csv")

print("Match features:", match_features.shape)
print("Player features:", player_features.shape)
print("Feature dictionary:", feature_dictionary.shape)

Match features: (7904, 26)
Player features: (274089, 7)
Feature dictionary: (21, 6)


In [41]:
match_features["match_date"] = pd.to_datetime(
    match_features["match_date"]
)

player_features["match_date"] = pd.to_datetime(
    player_features["match_date"]
)

In [42]:
def chronological_split(
    df,
    date_column="match_date",
    train_end_year=2022
):
    """
    Reproducible chronological split.

    Training:
        Seasons <= 2022

    Hold-out:
        Seasons > 2022
    """

    data = df.copy()

    data[date_column] = pd.to_datetime(data[date_column])

    train = data[
        data[date_column].dt.year <= train_end_year
    ].copy()

    holdout = data[
        data[date_column].dt.year > train_end_year
    ].copy()

    train = train.sort_values(date_column).reset_index(drop=True)
    holdout = holdout.sort_values(date_column).reset_index(drop=True)

    return train, holdout

In [43]:
train_matches, holdout_matches = chronological_split(
    match_features,
    date_column="match_date",
    train_end_year=2022
)

print("Training:", train_matches.shape)
print("Hold-out:", holdout_matches.shape)

print(
    "Training years:",
    train_matches["match_date"].dt.year.min(),
    "to",
    train_matches["match_date"].dt.year.max()
)

print(
    "Hold-out years:",
    holdout_matches["match_date"].dt.year.min(),
    "to",
    holdout_matches["match_date"].dt.year.max()
)

Training: (7256, 26)
Hold-out: (648, 26)
Training years: 1983 to 2022
Hold-out years: 2023 to 2025


# **Task 1**

### Home Win Baseline

In [44]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

In [45]:
y_true = holdout_matches["match_winner"]

# Always predict Home Win
home_win_predictions = np.full(
    len(holdout_matches),
    "Home Win"
)

home_win_accuracy = accuracy_score(
    y_true,
    home_win_predictions
)

home_win_f1 = f1_score(
    y_true,
    home_win_predictions,
    average="macro"
)

print("=== HOME-WIN BASELINE ===")
print("Accuracy:", round(home_win_accuracy, 4))
print("Macro F1:", round(home_win_f1, 4))

=== HOME-WIN BASELINE ===
Accuracy: 0.5664
Macro F1: 0.2411


### High Ladder Baseline

In [46]:
def higher_ladder_prediction(row):
    """
    Predict the team with the better pre-match ladder rank.
    Lower ladder rank means better position.
    """

    home_rank = row["home_pre_match_ladder_rank"]
    away_rank = row["away_pre_match_ladder_rank"]

    if home_rank < away_rank:
        return "Home Win"

    elif away_rank < home_rank:
        return "Away Win"

    else:
        # Deterministic tie-breaker
        return "Home Win"


ladder_predictions = holdout_matches.apply(
    higher_ladder_prediction,
    axis=1
)

ladder_accuracy = accuracy_score(
    y_true,
    ladder_predictions
)

ladder_f1 = f1_score(
    y_true,
    ladder_predictions,
    average="macro"
)

print("=== HIGHER-LADDER BASELINE ===")
print("Accuracy:", round(ladder_accuracy, 4))
print("Macro F1:", round(ladder_f1, 4))

=== HIGHER-LADDER BASELINE ===
Accuracy: 0.6219
Macro F1: 0.4157


### Match Winner Baselines Comparison

In [47]:
# Task 1A.3 — Match Winner Baseline Comparison

match_baseline_results = pd.DataFrame({
    "Model": [
        "Always Home Win",
        "Higher Ladder Team"
    ],
    "Accuracy": [
        home_win_accuracy,
        ladder_accuracy
    ],
    "Macro F1": [
        home_win_f1,
        ladder_f1
    ],
    "ROC AUC": [
        np.nan,
        np.nan
    ]
})

match_baseline_results

,Model,Accuracy,Macro F1,ROC AUC
0,Always Home Win,0.566358,0.241051,NaN
1,Higher Ladder Team,0.621914,0.415698,NaN


Baseline Models "Match Winner"

Two simple baselines were evaluated on the chronological hold-out period (2023–2025).

1. **Always Home Win:** predicts Home Win for every match.
2. **Higher Ladder Team:** predicts the team with the better pre-match ladder position.

The Always Home Win baseline achieved 56.64% accuracy and a Macro F1 of 0.2411. The lower Macro F1 reflects its inability to predict Away Wins and Draws.

The Higher Ladder Team baseline performed better, achieving 62.19% accuracy and a Macro F1 of 0.4157. This establishes the minimum benchmark that the trained match-winner models should improve upon.

ROC AUC was not calculated for these baselines because they do not generate meaningful class probability estimates.

### Top Player Baseline

In [48]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "players_round_by_round_stats_raw" in file:
            print(os.path.join(root, file))

/content/afl_datasets/afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv


In [49]:
player_round = pd.read_csv(
    "/content/afl_datasets/afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
)

player_round["match_date"] = pd.to_datetime(
    player_round["match_date"]
)

print("Shape:", player_round.shape)
print("Columns:")
print(player_round.columns.tolist())

Shape: (274089, 36)
Columns:
['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']


In [50]:
required_columns = [
    "player_id",
    "match_date",
    "year",
    "round",
    "team",
    "opponent",
    "disposals"
]

missing = [
    col for col in required_columns
    if col not in player_round.columns
]

print("Missing required columns:", missing)

Missing required columns: []


In [51]:
match_player_stats = player_round[
    [
        "player_id",
        "match_date",
        "year",
        "round",
        "team",
        "opponent",
        "disposals"
    ]
].copy()

match_player_stats = match_player_stats.dropna(
    subset=["disposals"]
)

match_player_stats = match_player_stats.sort_values(
    ["match_date", "round", "player_id"]
).reset_index(drop=True)

print("Rows with disposal data:", len(match_player_stats))

Rows with disposal data: 265636


In [52]:
match_max_disposals = (
    match_player_stats
    .groupby(
        ["match_date", "year", "round", "team", "opponent"]
    )["disposals"]
    .transform("max")
)

match_player_stats["is_leader"] = (
    match_player_stats["disposals"] == match_max_disposals
)

leader_counts = (
    match_player_stats
    .groupby(
        ["match_date", "year", "round", "team", "opponent"]
    )["is_leader"]
    .sum()
)

print(
    "Matches with a single disposal leader:",
    (leader_counts == 1).sum()
)

print(
    "Matches with tied disposal leaders:",
    (leader_counts > 1).sum()
)

Matches with a single disposal leader: 13244
Matches with tied disposal leaders: 1438


In [53]:
unique_leader_keys = leader_counts[
    leader_counts == 1
].index

unique_leaders = (
    match_player_stats
    .set_index(
        ["match_date", "year", "round", "team", "opponent"]
    )
    .loc[unique_leader_keys]
    .reset_index()
)

unique_leaders = unique_leaders[
    unique_leaders["is_leader"]
].copy()

unique_leaders = unique_leaders[
    [
        "match_date",
        "year",
        "round",
        "team",
        "opponent",
        "player_id",
        "disposals"
    ]
]

print("Unique leader matches:", len(unique_leaders))
print(unique_leaders.head())

Unique leader matches: 13244
  match_date  year round              team                   opponent  \
0 1983-03-27  1983     1  Essendon Bombers               Sydney Swans   
1 1983-04-04  1983     2  Essendon Bombers            St Kilda Saints   
2 1983-04-16  1983     4   St Kilda Saints               Geelong Cats   
3 1983-04-24  1983     5   St Kilda Saints               Sydney Swans   
4 1983-04-25  1983     5  Essendon Bombers  North Melbourne Kangaroos   

   player_id  disposals  
0      45852       10.0  
1      45852        8.0  
2      45679        5.0  
3      45679       10.0  
4      45852        1.0  


In [54]:
print(
    unique_leaders[
        ["match_date", "team", "opponent", "player_id", "disposals"]
    ].head(10)
)

  match_date              team                   opponent  player_id  \
0 1983-03-27  Essendon Bombers               Sydney Swans      45852   
1 1983-04-04  Essendon Bombers            St Kilda Saints      45852   
2 1983-04-16   St Kilda Saints               Geelong Cats      45679   
3 1983-04-24   St Kilda Saints               Sydney Swans      45679   
4 1983-04-25  Essendon Bombers  North Melbourne Kangaroos      45852   
5 1983-05-14  Essendon Bombers             Hawthorn Hawks      45852   
6 1983-05-21  Essendon Bombers           Melbourne Demons      45852   
7 1983-06-11  Essendon Bombers               Sydney Swans      45852   
8 1983-06-11   St Kilda Saints  North Melbourne Kangaroos      45679   
9 1983-06-18  Essendon Bombers            St Kilda Saints      45852   

   disposals  
0       10.0  
1        8.0  
2        5.0  
3       10.0  
4        1.0  
5       11.0  
6        4.0  
7        2.0  
8        6.0  
9       12.0  


In [55]:
# Sort historical leaders chronologically for each team
unique_leaders = unique_leaders.sort_values(
    ["team", "match_date", "round"]
).reset_index(drop=True)

# Previous disposal leader for each team
unique_leaders["previous_leader"] = (
    unique_leaders
    .groupby("team")["player_id"]
    .shift(1)
)

print(unique_leaders.head(10))

  match_date  year round            team                   opponent  \
0 1991-03-22  1991     1  Adelaide Crows             Hawthorn Hawks   
1 1991-03-31  1991     2  Adelaide Crows              Carlton Blues   
2 1991-04-07  1991     3  Adelaide Crows               Sydney Swans   
3 1991-04-13  1991     4  Adelaide Crows           Essendon Bombers   
4 1991-04-21  1991     5  Adelaide Crows          West Coast Eagles   
5 1991-04-28  1991     6  Adelaide Crows           Western Bulldogs   
6 1991-05-04  1991     7  Adelaide Crows            St Kilda Saints   
7 1991-05-17  1991     9  Adelaide Crows  North Melbourne Kangaroos   
8 1991-05-24  1991    10  Adelaide Crows           Melbourne Demons   
9 1991-06-01  1991    11  Adelaide Crows               Geelong Cats   

   player_id  disposals  previous_leader  
0      45884       15.0              NaN  
1      45884       13.0          45884.0  
2      45884       13.0          45884.0  
3      45353       16.0          45884.0  
4  

In [56]:
top_player_holdout = unique_leaders[
    unique_leaders["match_date"].dt.year > 2022
].copy()

print(
    "Hold-out unique-leader matches:",
    len(top_player_holdout)
)

print(
    "Hold-out date range:",
    top_player_holdout["match_date"].min(),
    "to",
    top_player_holdout["match_date"].max()
)

Hold-out unique-leader matches: 1165
Hold-out date range: 2023-03-16 00:00:00 to 2025-09-27 00:00:00


In [57]:
top_player_eval = top_player_holdout.dropna(
    subset=["previous_leader"]
).copy()

print(
    "Evaluable hold-out matches:",
    len(top_player_eval)
)

Evaluable hold-out matches: 1165


In [58]:
top_player_eval["top1_correct"] = (
    top_player_eval["previous_leader"]
    == top_player_eval["player_id"]
)

top1_accuracy = (
    top_player_eval["top1_correct"].mean()
)

print("=== TOP PLAYER BASELINE ===")
print(
    "Top-1 Accuracy:",
    round(top1_accuracy, 4)
)

=== TOP PLAYER BASELINE ===
Top-1 Accuracy: 0.2712


In [59]:
# Create a list of the previous 5 unique leaders for each team
def previous_five_leaders(group):
    leaders = group["player_id"].tolist()
    result = []

    for i in range(len(leaders)):
        previous = leaders[max(0, i - 5):i]
        result.append(previous)

    group = group.copy()
    group["previous_5_leaders"] = result

    return group


top_player_eval = (
    unique_leaders
    .sort_values(["team", "match_date", "round"])
    .groupby("team", group_keys=False)
    .apply(previous_five_leaders)
    .reset_index(drop=True)
)

# Keep hold-out only
top_player_eval = top_player_eval[
    top_player_eval["match_date"].dt.year > 2022
].copy()

# Remove matches where there is no historical leader
top_player_eval = top_player_eval[
    top_player_eval["previous_5_leaders"].apply(len) > 0
].copy()

top_player_eval["top5_hit"] = top_player_eval.apply(
    lambda row: row["player_id"] in row["previous_5_leaders"],
    axis=1
)

top5_hit_rate = top_player_eval["top5_hit"].mean()

print(
    "Top-5 Hit Rate:",
    round(top5_hit_rate, 4)
)

Top-5 Hit Rate: 0.6412


/tmp/ipykernel_4529/1817431414.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(previous_five_leaders)


In [60]:
top_player_baseline_results = pd.DataFrame({
    "Model": [
        "Previous Match Leader",
        "Previous 5 Leaders"
    ],
    "Metric": [
        "Top-1 Accuracy",
        "Top-5 Hit Rate"
    ],
    "Score": [
        top1_accuracy,
        top5_hit_rate
    ]
})

top_player_baseline_results

,Model,Metric,Score
0,Previous Match Leader,Top-1 Accuracy,0.271245
1,Previous 5 Leaders,Top-5 Hit Rate,0.641202


### Top Player Baselines

For top disposal player prediction, the baseline uses only historical information:

- **Previous Match Leader:** predicts that the team's disposal leader from its previous match will lead again.
- **Previous 5 Leaders:** checks whether the actual leader appears among the team's previous five unique disposal leaders.

The Previous Match Leader baseline achieved 27.12% Top-1 accuracy. The Previous 5 Leaders baseline achieved a 64.12% Top-5 hit rate.

These results provide reference points for evaluating the later top-player prediction model.

# TASK 2

In [61]:
# Load Day 1 match feature table
match_features = pd.read_csv("afl_match_features_v1.csv")

print("Shape:", match_features.shape)
print("\nAll columns:")
for i, col in enumerate(match_features.columns, 1):
    print(f"{i:2}. {col}")

print("\nTarget distribution:")
print(match_features["match_winner"].value_counts())

Shape: (7904, 26)

All columns:
 1. match_date
 2. round
 3. year
 4. home_team
 5. away_team
 6. venue
 7. match_winner
 8. home_recent_5_win_rate
 9. home_win_streak
10. home_recent_5_avg_score
11. home_days_rest
12. away_recent_5_win_rate
13. away_win_streak
14. away_recent_5_avg_score
15. away_days_rest
16. h2h_matches
17. h2h_current_home_wins
18. h2h_current_away_wins
19. h2h_draws
20. h2h_current_home_win_rate
21. home_pre_match_ladder_rank
22. home_pre_match_points
23. home_pre_match_percentage
24. away_pre_match_ladder_rank
25. away_pre_match_points
26. away_pre_match_percentage

Target distribution:
match_winner
Home Win    4669
Away Win    3170
Draw          65
Name: count, dtype: int64


In [62]:
# Leakage Check
target_column = "match_winner"

# Columns that should definitely NOT be used for prediction
excluded_columns = [
    "match_winner",
    "home_score",
    "away_score",
    "match_date",
    "year",
    "crowd"
]

print("Excluded columns:")
for col in excluded_columns:
    if col in match_features.columns:
        print(" -", col)

print("\nPotential modeling columns:")
candidate_features = [
    col for col in match_features.columns
    if col not in excluded_columns
]

for col in candidate_features:
    print(" -", col)

Excluded columns:
 - match_winner
 - match_date
 - year

Potential modeling columns:
 - round
 - home_team
 - away_team
 - venue
 - home_recent_5_win_rate
 - home_win_streak
 - home_recent_5_avg_score
 - home_days_rest
 - away_recent_5_win_rate
 - away_win_streak
 - away_recent_5_avg_score
 - away_days_rest
 - h2h_matches
 - h2h_current_home_wins
 - h2h_current_away_wins
 - h2h_draws
 - h2h_current_home_win_rate
 - home_pre_match_ladder_rank
 - home_pre_match_points
 - home_pre_match_percentage
 - away_pre_match_ladder_rank
 - away_pre_match_points
 - away_pre_match_percentage


### Train/Hold out Split

In [63]:
match_features["match_date"] = pd.to_datetime(match_features["match_date"])

# Training: seasons through 2022
# Hold-out: seasons 2023 onward
train_matches = (
    match_features[
        match_features["match_date"].dt.year <= 2022
    ]
    .sort_values("match_date")
    .reset_index(drop=True)
)

holdout_matches = (
    match_features[
        match_features["match_date"].dt.year > 2022
    ]
    .sort_values("match_date")
    .reset_index(drop=True)
)

print("Training set:")
print("Rows:", len(train_matches))
print("Date range:", train_matches["match_date"].min(), "to", train_matches["match_date"].max())

print("\nHold-out set:")
print("Rows:", len(holdout_matches))
print("Date range:", holdout_matches["match_date"].min(), "to", holdout_matches["match_date"].max())

print("\nClass distribution — Training:")
print(train_matches["match_winner"].value_counts())

print("\nClass distribution — Hold-out:")
print(holdout_matches["match_winner"].value_counts())

Training set:
Rows: 7256
Date range: 1983-03-26 00:00:00 to 2022-09-24 00:00:00

Hold-out set:
Rows: 648
Date range: 2023-03-16 00:00:00 to 2025-09-27 00:00:00

Class distribution — Training:
match_winner
Home Win    4302
Away Win    2895
Draw          59
Name: count, dtype: int64

Class distribution — Hold-out:
match_winner
Home Win    367
Away Win    275
Draw          6
Name: count, dtype: int64


### Define Features and Target

In [64]:
target_column = "match_winner"

feature_columns = [
    "round",
    "home_team",
    "away_team",
    "venue",

    "home_recent_5_win_rate",
    "home_win_streak",
    "home_recent_5_avg_score",
    "home_days_rest",

    "away_recent_5_win_rate",
    "away_win_streak",
    "away_recent_5_avg_score",
    "away_days_rest",

    "h2h_matches",
    "h2h_current_home_wins",
    "h2h_current_away_wins",
    "h2h_draws",
    "h2h_current_home_win_rate",

    "home_pre_match_ladder_rank",
    "home_pre_match_points",
    "home_pre_match_percentage",

    "away_pre_match_ladder_rank",
    "away_pre_match_points",
    "away_pre_match_percentage"
]

X_train = train_matches[feature_columns].copy()
y_train = train_matches[target_column].copy()

X_holdout = holdout_matches[feature_columns].copy()
y_holdout = holdout_matches[target_column].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_holdout shape:", X_holdout.shape)
print("y_holdout shape:", y_holdout.shape)

print("\nNumber of features:", len(feature_columns))

X_train shape: (7256, 23)
y_train shape: (7256,)
X_holdout shape: (648, 23)
y_holdout shape: (648,)

Number of features: 23


### Logistic Regression Pipeline

In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Categorical_features
categorical_features = [
    "round",
    "home_team",
    "away_team",
    "venue"
]

# Numeric Features
numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numeric features:", len(numeric_features))

print("\nNumeric features:")
for col in numeric_features:
    print(" -", col)

Categorical features:
['round', 'home_team', 'away_team', 'venue']

Number of categorical features: 4
Number of numeric features: 19

Numeric features:
 - home_recent_5_win_rate
 - home_win_streak
 - home_recent_5_avg_score
 - home_days_rest
 - away_recent_5_win_rate
 - away_win_streak
 - away_recent_5_avg_score
 - away_days_rest
 - h2h_matches
 - h2h_current_home_wins
 - h2h_current_away_wins
 - h2h_draws
 - h2h_current_home_win_rate
 - home_pre_match_ladder_rank
 - home_pre_match_points
 - home_pre_match_percentage
 - away_pre_match_ladder_rank
 - away_pre_match_points
 - away_pre_match_percentage


Column Transformer

In [66]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

print("ColumnTransformer created successfully.")

ColumnTransformer created successfully.


Complete Pipeline

In [19]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully.")
print(logistic_pipeline)

Logistic Regression pipeline created successfully.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['home_recent_5_win_rate',
                                                   'home_win_streak',
                                                   'home_recent_5_avg_score',
                                                   'home_days_rest',
                                                   'away_recent_5_win_rate',
                                                   'away_win_streak',
                                                   'away_recent_5_avg

Training Model

In [67]:
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


### Evaluate On Hold Out

In [68]:
# Predicted classes
logistic_predictions = logistic_pipeline.predict(X_holdout)

# Predicted probabilities
logistic_probabilities = logistic_pipeline.predict_proba(X_holdout)

# Classes corresponding to probability columns
logistic_classes = logistic_pipeline.named_steps["model"].classes_

print("Classes:")
print(logistic_classes)

print("\nPrediction shape:", logistic_predictions.shape)
print("Probability shape:", logistic_probabilities.shape)

print("\nFirst 5 predictions:")
print(logistic_predictions[:5])

print("\nFirst 5 probability rows:")
print(logistic_probabilities[:5])

Classes:
['Away Win' 'Draw' 'Home Win']

Prediction shape: (648,)
Probability shape: (648, 3)

First 5 predictions:
['Home Win' 'Home Win' 'Away Win' 'Home Win' 'Home Win']

First 5 probability rows:
[[4.23693224e-01 1.62990081e-02 5.60007768e-01]
 [2.91056804e-01 2.08054041e-04 7.08735142e-01]
 [6.21615988e-01 5.60797511e-03 3.72776036e-01]
 [4.35948451e-01 2.84615641e-03 5.61205392e-01]
 [4.38749566e-01 1.06488439e-03 5.60185550e-01]]


In [69]:
logistic_accuracy = accuracy_score(
    y_holdout,
    logistic_predictions
)

logistic_f1 = f1_score(
    y_holdout,
    logistic_predictions,
    average="macro"
)

logistic_roc_auc = roc_auc_score(
    y_holdout,
    logistic_probabilities,
    multi_class="ovr",
    average="macro"
)

print(f"Logistic Regression Accuracy : {logistic_accuracy:.4f}")
print(f"Logistic Regression Macro F1: {logistic_f1:.4f}")
print(f"Logistic Regression ROC AUC : {logistic_roc_auc:.4f}")

Logistic Regression Accuracy : 0.6373
Logistic Regression Macro F1: 0.4204
Logistic Regression ROC AUC : 0.6084


Calibration(Multiclass Briar Score)

In [70]:
# Convert actual labels into one-hot format
y_holdout_onehot = pd.get_dummies(
    y_holdout
).reindex(
    columns=logistic_classes,
    fill_value=0
).astype(int).values

# Multiclass Brier score
logistic_brier = np.mean(
    np.sum(
        (y_holdout_onehot - logistic_probabilities) ** 2,
        axis=1
    )
)

print(f"Logistic Regression Multiclass Brier Score: {logistic_brier:.4f}")

Logistic Regression Multiclass Brier Score: 0.4467


In [78]:

logistic_results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [logistic_accuracy],
    "Macro F1": [logistic_f1],
    "ROC AUC": [logistic_roc_auc],
    "Brier Score": [logistic_brier]
})

logistic_results

,Model,Accuracy,Macro F1,ROC AUC,Brier Score
0,Logistic Regression,0.637346,0.420392,0.608449,0.446745


### Gradiat Boosting Pipeline

In [71]:
from sklearn.ensemble import GradientBoostingClassifier

gb_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

gb_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

gb_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", gb_numeric_pipeline, numeric_features),
        ("categorical", gb_categorical_pipeline, categorical_features)
    ]
)

# Complete Pipeline
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", gb_preprocessor),
        (
            "model",
            GradientBoostingClassifier(
                random_state=42,
                n_estimators=150,
                learning_rate=0.05,
                max_depth=3
            )
        )
    ]
)

print("Gradient Boosting pipeline created successfully.")

Gradient Boosting pipeline created successfully.


Training Model

In [72]:
gb_pipeline.fit(X_train, y_train)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


GB Hold out Prediction

In [73]:
gb_predictions = gb_pipeline.predict(X_holdout)

gb_probabilities = gb_pipeline.predict_proba(X_holdout)

gb_classes = gb_pipeline.named_steps["model"].classes_

print("Classes:")
print(gb_classes)

print("\nPrediction shape:", gb_predictions.shape)
print("Probability shape:", gb_probabilities.shape)

print("\nFirst 5 predictions:")
print(gb_predictions[:5])

print("\nFirst 5 probability rows:")
print(gb_probabilities[:5])

Classes:
['Away Win' 'Draw' 'Home Win']

Prediction shape: (648,)
Probability shape: (648, 3)

First 5 predictions:
['Home Win' 'Home Win' 'Home Win' 'Home Win' 'Home Win']

First 5 probability rows:
[[0.36749867 0.00661563 0.6258857 ]
 [0.32218489 0.00518715 0.67262795]
 [0.48486648 0.00633481 0.50879871]
 [0.41028836 0.01011943 0.57959221]
 [0.47430854 0.00509941 0.52059205]]


Classification Metrics

In [74]:
gb_accuracy = accuracy_score(
    y_holdout,
    gb_predictions
)

gb_f1 = f1_score(
    y_holdout,
    gb_predictions,
    average="macro"
)

gb_roc_auc = roc_auc_score(
    y_holdout,
    gb_probabilities,
    multi_class="ovr",
    average="macro"
)

print(f"Gradient Boosting Accuracy : {gb_accuracy:.4f}")
print(f"Gradient Boosting Macro F1: {gb_f1:.4f}")
print(f"Gradient Boosting ROC AUC : {gb_roc_auc:.4f}")

Gradient Boosting Accuracy : 0.6404
Gradient Boosting Macro F1: 0.4207
Gradient Boosting ROC AUC : 0.6508


Calibration(Briar Score)

In [75]:
y_holdout_onehot_gb = pd.get_dummies(
    y_holdout
).reindex(
    columns=gb_classes,
    fill_value=0
).astype(int).values

gb_brier = np.mean(
    np.sum(
        (y_holdout_onehot_gb - gb_probabilities) ** 2,
        axis=1
    )
)

print(f"Gradient Boosting Multiclass Brier Score: {gb_brier:.4f}")

Gradient Boosting Multiclass Brier Score: 0.4281


### Comparision Table

In [79]:
gb_results = pd.DataFrame({
    "Model": ["Gradient Boosting"],
    "Accuracy": [gb_accuracy],
    "Macro F1": [gb_f1],
    "ROC AUC": [gb_roc_auc],
    "Brier Score": [gb_brier]
})

model_comparison = pd.concat(
    [logistic_results, gb_results],
    ignore_index=True
)

model_comparison

,Model,Accuracy,Macro F1,ROC AUC,Brier Score
0,Logistic Regression,0.637346,0.420392,0.608449,0.446745
1,Gradient Boosting,0.640432,0.420682,0.650809,0.428085


### Saving Comparision Table

In [80]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Gradient Boosting"
    ],
    "Accuracy": [
        logistic_accuracy,
        gb_accuracy
    ],
    "Macro F1": [
        logistic_f1,
        gb_f1
    ],
    "ROC AUC": [
        logistic_roc_auc,
        gb_roc_auc
    ],
    "Brier Score": [
        logistic_brier,
        gb_brier
    ]
})

model_comparison.round(4)

,Model,Accuracy,Macro F1,ROC AUC,Brier Score
0,Logistic Regression,0.6373,0.4204,0.6084,0.4467
1,Gradient Boosting,0.6404,0.4207,0.6508,0.4281


### Final Model Selection

Gradient Boosting was selected as the final match-winner prediction model.

On the chronological hold-out set, Gradient Boosting achieved an accuracy of 0.6404, macro F1 of 0.4207, ROC AUC of 0.6508, and Brier score of 0.4281. Logistic Regression achieved 0.6373 accuracy, 0.4204 macro F1, 0.6084 ROC AUC, and 0.4467 Brier score.

Gradient Boosting performed better on all four metrics. The largest improvement was in ROC AUC, indicating better separation between match-outcome classes. Its lower Brier score also indicates better probability quality.

The main trade-off is interpretability. Logistic Regression is more transparent because its coefficients directly show the direction and magnitude of feature effects. Gradient Boosting is less directly interpretable because it combines many decision trees and can model non-linear relationships and feature interactions. However, feature importance can still be extracted from the trained Gradient Boosting model to understand which variables contribute most to predictions.

Therefore, Gradient Boosting is retained as the final predictive model, while Logistic Regression serves as an interpretable benchmark.

# **TASK 3**

### Top Player Model Framing

The top player problem is framed as a regression task followed by ranking.

The model predicts each player's expected disposals for their upcoming match using only pre-match information. Players are then ranked by their predicted disposals, and the highest-ranked players form the predicted top-player list.

Regression was selected because disposals are a continuous numerical player statistic that can be predicted directly. This approach also provides interpretable error metrics such as MAE and RMSE while still supporting the final ranking use case.

The model is evaluated using MAE, RMSE, and top-5 hit rate. The top-5 hit rate measures whether the actual top-disposal player was included among the five highest predicted players.

This approach is compared against the Task 1 baseline, where the previous match's disposal leader is used as the prediction. The model is considered meaningfully better if it improves the top-5 hit rate while maintaining reasonable regression error.

In [81]:
# Inspect the player feature dataset

print("Shape:", player_features.shape)
print("\nColumns:")
print(player_features.columns.tolist())

print("\nFirst 5 rows:")
display(player_features.head())

Shape: (274089, 7)

Columns:
['player_id', 'match_date', 'year', 'round', 'team', 'opponent', 'player_recent_5_avg_disposals']

First 5 rows:


,player_id,match_date,year,round,team,opponent,player_recent_5_avg_disposals
0,43260,2020-07-05,2020,5,Richmond Tigers,Melbourne Demons,NaN
1,43260,2020-07-12,2020,6,Richmond Tigers,Sydney Swans,16.00
2,43260,2020-07-18,2020,7,Richmond Tigers,North Melbourne Kangaroos,11.00
3,43260,2020-07-24,2020,8,Richmond Tigers,Greater Western Sydney Giants,10.00
4,43260,2020-07-29,2020,9,Richmond Tigers,Western Bulldogs,8.75


In [82]:
# Check data types and missing values

print("Data types:")
display(player_features.dtypes)

print("\nMissing values:")
display(
    player_features.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

Data types:


,0
player_id,int64
match_date,datetime64[ns]
year,int64
round,object
team,object
opponent,object
player_recent_5_avg_disposals,float64



Missing values:


,missing_count
player_recent_5_avg_disposals,3229
match_date,0
player_id,0
year,0
round,0
team,0
opponent,0


### Building Regression Dataset

In [83]:
# Load the raw player round-by-round statistics

player_stats = pd.read_csv(
    "afl_datasets/afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
)

player_stats["match_date"] = pd.to_datetime(player_stats["match_date"])

print("Shape:", player_stats.shape)
print("\nColumns:")
print(player_stats.columns.tolist())

display(player_stats.head())

Shape: (274089, 36)

Columns:
['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']


,id,team,year,career_game_count,opponent,round,result,jersey_num,kicks,marks,...,marks_inside_50,one_percenters,bounces,goal_assist,percentage_of_game_played,player_id,match_date,fantasy_points,score,margin
0,556392,Hawthorn Hawks,1994,17,Richmond Tigers,21,W,34,5.0,4.0,...,NaN,NaN,NaN,NaN,NaN,45552,1994-08-14,36,NaN,28
1,614897,Geelong Cats,2024,1,St Kilda Saints,1,W,7,5.0,NaN,...,NaN,1.0,NaN,NaN,26.0,44356,2024-03-16,23,NaN,8
2,583553,Essendon Bombers,1999,97,Adelaide Crows,10,W,6,14.0,5.0,...,NaN,3.0,NaN,NaN,NaN,45955,1999-06-04,67,NaN,48
3,590676,Western Bulldogs,1994,36,St Kilda Saints,21,W,35,12.0,10.0,...,NaN,NaN,NaN,NaN,NaN,45656,1994-08-13,81,NaN,45
4,582473,Richmond Tigers,1997,113,Melbourne Demons,10,L,41,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,45929,1997-05-31,32,NaN,-25


In [84]:
# Verify the disposals target

print("Missing disposals:", player_stats["disposals"].isna().sum())
print("Total rows:", len(player_stats))

print("\nDisposals summary:")
display(player_stats["disposals"].describe())

Missing disposals: 8453
Total rows: 274089

Disposals summary:


,disposals
count,265636.000000
mean,15.070299
std,7.304748
min,-5.000000
25%,10.000000
50%,14.000000
75%,20.000000
max,54.000000


In [85]:
# Recreate the feature and target tables from the original sources

merge_keys = [
    "player_id",
    "match_date",
    "team",
    "opponent",
    "round"
]

feature_data = player_features[
    merge_keys + ["player_recent_5_avg_disposals"]
].copy()

target_data = player_stats[
    merge_keys + ["disposals"]
].copy()

# Add occurrence number within each merge key.
# This prevents many-to-many multiplication for the 10 duplicated keys.

feature_data["_occurrence"] = (
    feature_data.groupby(merge_keys).cumcount()
)

target_data["_occurrence"] = (
    target_data.groupby(merge_keys).cumcount()
)

# Merge using the occurrence number as part of the key

player_model_data = feature_data.merge(
    target_data,
    on=merge_keys + ["_occurrence"],
    how="inner",
    validate="one_to_one"
)

# Remove the temporary occurrence column
player_model_data = player_model_data.drop(columns="_occurrence")

print("Merged rows:", len(player_model_data))
print("Expected rows:", len(player_features))

print("\nMissing disposals:", player_model_data["disposals"].isna().sum())

display(
    player_model_data.head(10)
)

Merged rows: 274089
Expected rows: 274089

Missing disposals: 8453


,player_id,match_date,team,opponent,round,player_recent_5_avg_disposals,disposals
0,43260,2020-07-05,Richmond Tigers,Melbourne Demons,5,NaN,16.0
1,43260,2020-07-12,Richmond Tigers,Sydney Swans,6,16.00,6.0
2,43260,2020-07-18,Richmond Tigers,North Melbourne Kangaroos,7,11.00,8.0
3,43260,2020-07-24,Richmond Tigers,Greater Western Sydney Giants,8,10.00,5.0
4,43260,2020-07-29,Richmond Tigers,Western Bulldogs,9,8.75,7.0
5,43260,2020-08-04,Richmond Tigers,Brisbane Lions,10,8.40,14.0
6,43260,2020-08-08,Richmond Tigers,Port Adelaide Power,11,8.00,10.0
7,43260,2020-08-17,Richmond Tigers,Gold Coast Suns,12,8.80,4.0
8,43260,2020-08-22,Richmond Tigers,Essendon Bombers,13,8.00,10.0
9,43260,2020-08-27,Richmond Tigers,West Coast Eagles,14,9.00,16.0


In [86]:
# Verify that the corrected merge has not created extra rows

print("Row count preserved:",
      len(player_model_data) == len(player_features))

print("Duplicate merge keys after pairing:")

remaining_duplicates = player_model_data.duplicated(
    subset=merge_keys,
    keep=False
)

print("Rows with duplicated keys:", remaining_duplicates.sum())

print(
    "Number of duplicated keys:",
    player_model_data.loc[
        remaining_duplicates,
        merge_keys
    ].drop_duplicates().shape[0]
)

Row count preserved: True
Duplicate merge keys after pairing:
Rows with duplicated keys: 20
Number of duplicated keys: 10


In [87]:
# Keep only observations with a known disposal target

player_model_data = player_model_data[
    player_model_data["disposals"].notna()
].copy()

print("Usable modeling rows:", len(player_model_data))
print("Missing targets:", player_model_data["disposals"].isna().sum())

print("\nTarget statistics:")
display(player_model_data["disposals"].describe())

Usable modeling rows: 265636
Missing targets: 0

Target statistics:


,disposals
count,265636.000000
mean,15.070299
std,7.304748
min,-5.000000
25%,10.000000
50%,14.000000
75%,20.000000
max,54.000000


In [88]:
# Convert date and create chronological train/hold-out split

player_model_data["match_date"] = pd.to_datetime(
    player_model_data["match_date"]
)

player_train = player_model_data[
    player_model_data["match_date"].dt.year <= 2022
].copy()

player_holdout = player_model_data[
    player_model_data["match_date"].dt.year > 2022
].copy()

# Sort chronologically
player_train = player_train.sort_values("match_date").reset_index(drop=True)
player_holdout = player_holdout.sort_values("match_date").reset_index(drop=True)

print("Training rows:", len(player_train))
print("Hold-out rows:", len(player_holdout))

print("\nTraining period:")
print(
    player_train["match_date"].min(),
    "to",
    player_train["match_date"].max()
)

print("\nHold-out period:")
print(
    player_holdout["match_date"].min(),
    "to",
    player_holdout["match_date"].max()
)

Training rows: 236614
Hold-out rows: 29022

Training period:
1983-03-27 00:00:00 to 2022-09-24 00:00:00

Hold-out period:
2023-03-16 00:00:00 to 2025-09-27 00:00:00


In [89]:
# Verify the chronological split

print("Training years:",
      player_train["match_date"].dt.year.min(),
      "to",
      player_train["match_date"].dt.year.max())

print("Hold-out years:",
      player_holdout["match_date"].dt.year.min(),
      "to",
      player_holdout["match_date"].dt.year.max())

print("\nTarget statistics:")

target_summary = pd.DataFrame({
    "Train": player_train["disposals"].describe(),
    "Holdout": player_holdout["disposals"].describe()
})

display(target_summary)

Training years: 1983 to 2022
Hold-out years: 2023 to 2025

Target statistics:


,Train,Holdout
count,236614.000000,29022.000000
mean,15.056332,15.184171
std,7.286089,7.454278
min,-5.000000,-5.000000
25%,10.000000,10.000000
50%,14.000000,14.000000
75%,20.000000,20.000000
max,54.000000,54.000000


### Regression Features

In [90]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Predictor and target
player_feature_columns = [
    "player_recent_5_avg_disposals"
]

X_player_train = player_train[player_feature_columns]
y_player_train = player_train["disposals"]

X_player_holdout = player_holdout[player_feature_columns]
y_player_holdout = player_holdout["disposals"]

print("X train shape:", X_player_train.shape)
print("X holdout shape:", X_player_holdout.shape)
print("y train shape:", y_player_train.shape)
print("y holdout shape:", y_player_holdout.shape)

X train shape: (236614, 1)
X holdout shape: (29022, 1)
y train shape: (236614,)
y holdout shape: (29022,)


### Train Random Forest Regression

In [91]:
# Random Forest regression pipeline

player_regression_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )
])

player_regression_pipeline.fit(
    X_player_train,
    y_player_train
)

print("Top-player regression model trained successfully.")

Top-player regression model trained successfully.


In [92]:
# Predict player disposals on the chronological hold-out set

player_predictions = player_regression_pipeline.predict(
    X_player_holdout
)

player_mae = mean_absolute_error(
    y_player_holdout,
    player_predictions
)

player_rmse = np.sqrt(
    mean_squared_error(
        y_player_holdout,
        player_predictions
    )
)

print(f"MAE:  {player_mae:.4f}")
print(f"RMSE: {player_rmse:.4f}")

MAE:  4.0486
RMSE: 5.1759


TOP 5 hit Rate

In [93]:
# Add predictions to the holdout data
player_eval = player_holdout[
    ["player_id", "match_date", "team", "opponent", "disposals"]
].copy()

player_eval["predicted_disposals"] = player_predictions

# Rank players within each match by predicted disposals
player_eval["predicted_rank"] = (
    player_eval.groupby(["match_date", "team", "opponent"])["predicted_disposals"]
    .rank(method="first", ascending=False)
)

# Rank players within each match by actual disposals
player_eval["actual_rank"] = (
    player_eval.groupby(["match_date", "team", "opponent"])["disposals"]
    .rank(method="min", ascending=False)
)

# Evaluate only matches where an actual unique disposal leader exists
actual_leaders = (
    player_eval[player_eval["actual_rank"] == 1]
    .groupby(["match_date", "team", "opponent"])
    .size()
)

unique_leader_matches = actual_leaders[actual_leaders == 1].index

player_eval_unique = player_eval.set_index(
    ["match_date", "team", "opponent"]
).loc[unique_leader_matches].reset_index()

# Predicted top-5 players
predicted_top5 = (
    player_eval_unique[player_eval_unique["predicted_rank"] <= 5]
    .groupby(["match_date", "team", "opponent"])["player_id"]
    .apply(set)
)

# Actual disposal leader
actual_top1 = (
    player_eval_unique[player_eval_unique["actual_rank"] == 1]
    .groupby(["match_date", "team", "opponent"])["player_id"]
    .first()
)

# Top-5 hit rate
top5_hits = []

for match_key in actual_top1.index:
    actual_player = actual_top1.loc[match_key]
    predicted_players = predicted_top5.loc[match_key]
    top5_hits.append(actual_player in predicted_players)

top5_hit_rate = np.mean(top5_hits)

print(f"Unique-leader holdout matches: {len(top5_hits):,}")
print(f"Top-5 Hit Rate: {top5_hit_rate:.4f}")
print(f"Top-5 Hit Rate: {top5_hit_rate:.2%}")

Unique-leader holdout matches: 1,165
Top-5 Hit Rate: 0.8635
Top-5 Hit Rate: 86.35%


### Evaluation and Baseline Comparison

The top-player problem was framed as **regression followed by ranking** rather than direct learning-to-rank. A Random Forest Regressor predicts each player's expected disposals, and players are then ranked within each match according to their predicted disposals.

On the chronological 2023–2025 holdout set:

- MAE: **4.0486**
- RMSE: **5.1759**
- Top-5 Hit Rate: **86.35%**

The Day 2 Task 1 baseline, based on previous player leaders, achieved a Top-5 Hit Rate of **64.12%**.

The regression model therefore improved Top-5 Hit Rate by **22.23 percentage points**, from 64.12% to 86.35%. This is a meaningful improvement over the baseline.

Because the selected framing is regression --> ranking, MAE, RMSE, and Top-K Hit Rate are the primary evaluation metrics. NDCG is not required because a separate learning-to-rank model was not used.

# TASK 4

### Match Winner Feature Importance

In [94]:
# Get transformed feature names from the fitted preprocessing pipeline
gb_preprocessor_fitted = gb_pipeline.named_steps["preprocessor"]
gb_model = gb_pipeline.named_steps["model"]

gb_feature_names = gb_preprocessor_fitted.get_feature_names_out()

gb_importance = pd.DataFrame({
    "feature": gb_feature_names,
    "importance": gb_model.feature_importances_
}).sort_values("importance", ascending=False)

gb_importance.head(20)

,feature,importance
15,numeric__home_pre_match_percentage,0.280406
18,numeric__away_pre_match_percentage,0.262159
14,numeric__home_pre_match_points,0.067849
0,numeric__home_recent_5_win_rate,0.037463
6,numeric__away_recent_5_avg_score,0.036832
17,numeric__away_pre_match_points,0.032786
9,numeric__h2h_current_home_wins,0.025767
4,numeric__away_recent_5_win_rate,0.022723
2,numeric__home_recent_5_avg_score,0.022560
8,numeric__h2h_matches,0.020773


In [95]:
gb_importance_display = gb_importance.head(15).copy()

gb_importance_display["feature"] = (
    gb_importance_display["feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

gb_importance_display

,feature,importance
15,home_pre_match_percentage,0.280406
18,away_pre_match_percentage,0.262159
14,home_pre_match_points,0.067849
0,home_recent_5_win_rate,0.037463
6,away_recent_5_avg_score,0.036832
17,away_pre_match_points,0.032786
9,h2h_current_home_wins,0.025767
4,away_recent_5_win_rate,0.022723
2,home_recent_5_avg_score,0.022560
8,h2h_matches,0.020773


### Player Model Feature Importance

In [96]:
# Extract Random Forest feature importance
rf_model = player_regression_pipeline.named_steps["model"]

player_importance = pd.DataFrame({
    "feature": player_feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

player_importance

,feature,importance
0,player_recent_5_avg_disposals,1.0


### Player Model Feature Importance

The Random Forest regression model assigns an importance of **1.0000 (100%)** to `player_recent_5_avg_disposals`.

This is expected because the current model uses only one predictor. The feature is football-sensible because a player's recent disposal production is directly relevant to predicting their disposal output in an upcoming match.

No obvious target leakage is present because `player_recent_5_avg_disposals` is calculated from the player's **previous five matches** and does not include the target match's disposal value.

A limitation is that the current player model has a very small feature set. Therefore, this importance analysis confirms the model's dependence on recent form but cannot evaluate the contribution of other factors such as opponent, venue, team context, rest, or career/season form.

### Sniff Test

In [97]:
# Match features used by the Gradient Boosting model

match_feature_columns = [
    "round",
    "home_team",
    "away_team",
    "venue",
    "home_recent_5_win_rate",
    "home_win_streak",
    "home_recent_5_avg_score",
    "home_days_rest",
    "away_recent_5_win_rate",
    "away_win_streak",
    "away_recent_5_avg_score",
    "away_days_rest",
    "h2h_matches",
    "h2h_current_home_wins",
    "h2h_current_away_wins",
    "h2h_draws",
    "h2h_current_home_win_rate",
    "home_pre_match_ladder_rank",
    "home_pre_match_points",
    "home_pre_match_percentage",
    "away_pre_match_ladder_rank",
    "away_pre_match_points",
    "away_pre_match_percentage"
]

print("Number of match features:", len(match_feature_columns))

Number of match features: 23


In [98]:
# Create a copy of the match holdout data
match_sniff = holdout_matches.copy()

# Generate model predictions
match_sniff["predicted_winner"] = gb_pipeline.predict(
    match_sniff[match_feature_columns]
)

# Generate probabilities
match_probabilities = gb_pipeline.predict_proba(
    match_sniff[match_feature_columns]
)

# Get class names
class_names = gb_pipeline.named_steps["model"].classes_

for i, class_name in enumerate(class_names):
    match_sniff[f"prob_{class_name}"] = match_probabilities[:, i]

# Prediction confidence
match_sniff["prediction_confidence"] = match_probabilities.max(axis=1)

# Check whether prediction was correct
match_sniff["correct"] = (
    match_sniff["predicted_winner"] == match_sniff["match_winner"]
)

print("Holdout matches:", len(match_sniff))

Holdout matches: 648


### Three representative Matches for Sniff test

In [99]:
# 1. Most confident correct prediction
confident_correct = (
    match_sniff[match_sniff["correct"]]
    .sort_values("prediction_confidence", ascending=False)
    .head(1)
)

# 2. Most confident incorrect prediction
confident_incorrect = (
    match_sniff[~match_sniff["correct"]]
    .sort_values("prediction_confidence", ascending=False)
    .head(1)
)

# 3. A moderate-confidence prediction
moderate_confidence = (
    match_sniff
    .sort_values("prediction_confidence")
    .iloc[[len(match_sniff) // 2]]
)

sniff_matches = pd.concat([
    confident_correct,
    confident_incorrect,
    moderate_confidence
])

display(
    sniff_matches[[
        "match_date",
        "round",
        "home_team",
        "away_team",
        "match_winner",
        "predicted_winner",
        "prediction_confidence"
    ]]
)

,match_date,round,home_team,away_team,match_winner,predicted_winner,prediction_confidence
612,2025-08-08,22,Geelong Cats,Essendon Bombers,Home Win,Home Win,0.916523
195,2023-08-20,23,Western Bulldogs,West Coast Eagles,Away Win,Home Win,0.887457
83,2023-05-20,10,North Melbourne Kangaroos,Sydney Swans,Away Win,Away Win,0.638144


In [100]:
# Inspect the pre match information for the 3 sniff test matches

sniff_feature_columns = [
    "match_date",
    "round",
    "home_team",
    "away_team",
    "home_recent_5_win_rate",
    "away_recent_5_win_rate",
    "home_win_streak",
    "away_win_streak",
    "home_recent_5_avg_score",
    "away_recent_5_avg_score",
    "home_days_rest",
    "away_days_rest",
    "h2h_matches",
    "h2h_current_home_wins",
    "h2h_current_away_wins",
    "h2h_draws",
    "h2h_current_home_win_rate",
    "home_pre_match_ladder_rank",
    "away_pre_match_ladder_rank",
    "home_pre_match_points",
    "away_pre_match_points",
    "home_pre_match_percentage",
    "away_pre_match_percentage",
    "predicted_winner",
    "prediction_confidence",
    "match_winner"
]

display(
    sniff_matches[sniff_feature_columns]
    .T
)

,612,195,83
match_date,2025-08-08 00:00:00,2023-08-20 00:00:00,2023-05-20 00:00:00
round,22,23,10
home_team,Geelong Cats,Western Bulldogs,North Melbourne Kangaroos
away_team,Essendon Bombers,West Coast Eagles,Sydney Swans
home_recent_5_win_rate,0.8,0.4,0.0
away_recent_5_win_rate,0.0,0.2,0.2
home_win_streak,3,0,0
away_win_streak,0,0,0
home_recent_5_avg_score,125.0,85.8,55.8
away_recent_5_avg_score,54.2,61.2,79.8


### Sniff Test — Held-Out Matches

Three matches from the 2023–2025 holdout set were selected for a manual sanity check. The expected outcome was reasoned from pre-match information only, including recent form, recent scoring, ladder position, points, percentage, H2H history, and rest days.

| Match | Manual Expectation | Model Prediction | Confidence | Actual Result |
|---|---|---|---:|---|
| Geelong Cats vs Essendon Bombers | Home Win | Home Win | 91.65% | Home Win |
| Western Bulldogs vs West Coast Eagles | Home Win | Home Win | 88.75% | Away Win |
| North Melbourne Kangaroos vs Sydney Swans | Away Win | Away Win | 63.81% | Away Win |

The Geelong vs Essendon case showed strong agreement. Geelong had substantially better recent form, recent scoring, ladder position, points, and percentage, so a Home Win was reasonable. The model correctly predicted the outcome with 91.65% confidence.

The Western Bulldogs vs West Coast case was the major disagreement. Both manual reasoning and the model favored the Bulldogs because the Bulldogs had better recent form, scoring, ladder position, points, and percentage. However, West Coast won. The disagreement appears to be a legitimate prediction error rather than evidence of target leakage because only pre-match features were used. It highlights that the current feature set cannot capture every factor contributing to an individual match outcome or upset.

The North Melbourne vs Sydney case also showed agreement. Sydney had better recent scoring, recent win rate, ladder position, and percentage. The model correctly predicted an Away Win with 63.81% confidence.

Overall, the sniff test supports the football plausibility of the model's predictions and important features. The main disagreement was explainable as an ordinary prediction error rather than suspicious leakage.

# **TASK 5**

### Saving Trained Pipelines

In [101]:
import joblib
import os

# Create a directory for model artifacts
os.makedirs("model_artifacts", exist_ok=True)

# Save match winner model
joblib.dump(
    gb_pipeline,
    "model_artifacts/match_winner_pipeline.joblib"
)

# Save top-player regression model
joblib.dump(
    player_regression_pipeline,
    "model_artifacts/top_player_pipeline.joblib"
)

print("Saved model artifacts:")
print(os.listdir("model_artifacts"))

Saved model artifacts:
['top_player_pipeline.joblib', 'match_winner_pipeline.joblib']


### Predict Match Winner

In [102]:
import pandas as pd
import numpy as np

def predict_match_winner(team_a, team_b, date):
    """
    Predict the winner of an AFL match using the trained Gradient Boosting model.

    Parameters:
    team_a : str
        Home team name.
    team_b : str
        Away team name.
    date : str or datetime
        Match date.

    Returns:
    dict
        Prediction containing teams, winner, and probability.
    """

    # 1. Validate team inputs
    if not isinstance(team_a, str) or not team_a.strip():
        raise ValueError("team_a must be a non-empty team name.")

    if not isinstance(team_b, str) or not team_b.strip():
        raise ValueError("team_b must be a non-empty team name.")

    team_a = team_a.strip()
    team_b = team_b.strip()

    if team_a == team_b:
        raise ValueError("team_a and team_b must be different teams.")


    # 2. Validate date
    try:
        match_date = pd.to_datetime(date)
    except Exception:
        raise ValueError(
            "Invalid date. Please provide a valid date such as '2025-08-08'."
        )


    # 3. Validate team names
    known_home_teams = set(match_features["home_team"].dropna().unique())
    known_away_teams = set(match_features["away_team"].dropna().unique())
    known_teams = known_home_teams.union(known_away_teams)

    if team_a not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team_a}'. "
            f"Please use a team name from the dataset."
        )

    if team_b not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team_b}'. "
            f"Please use a team name from the dataset."
        )

    # 4. Validate date range
    min_date = pd.to_datetime(match_features["match_date"]).min()
    max_date = pd.to_datetime(match_features["match_date"]).max()

    if match_date < min_date or match_date > max_date:
        raise ValueError(
            f"Date '{match_date.date()}' is outside the available "
            f"data range ({min_date.date()} to {max_date.date()})."
        )

    # 5. Find matching feature row
    data = match_features.copy()
    data["match_date"] = pd.to_datetime(data["match_date"])

    match_row = data[
        (data["match_date"] == match_date) &
        (data["home_team"] == team_a) &
        (data["away_team"] == team_b)
    ]

    if match_row.empty:
        raise ValueError(
            f"No match found for {team_a} vs {team_b} "
            f"on {match_date.date()} in the feature dataset."
        )

    # 6. Prepare model input
    X = match_row[match_feature_columns]

    # 7. Predict
    prediction = gb_pipeline.predict(X)[0]
    probabilities = gb_pipeline.predict_proba(X)[0]

    class_names = gb_pipeline.named_steps["model"].classes_

    probability_map = dict(zip(class_names, probabilities))

    return {
        "home_team": team_a,
        "away_team": team_b,
        "date": str(match_date.date()),
        "winner": prediction,
        "probability": round(float(probability_map[prediction]), 4),
        "class_probabilities": {
            cls: round(float(prob), 4)
            for cls, prob in probability_map.items()
        }
    }

In [103]:
result = predict_match_winner(
    "Geelong Cats",
    "Essendon Bombers",
    "2025-08-08"
)

print(result)

{'home_team': 'Geelong Cats', 'away_team': 'Essendon Bombers', 'date': '2025-08-08', 'winner': 'Home Win', 'probability': 0.9165, 'class_probabilities': {'Away Win': 0.0795, 'Draw': 0.004, 'Home Win': 0.9165}}


Validation Test

In [104]:
# Test 1: Unknown home team
try:
    predict_match_winner(
        "Unknown FC",
        "Essendon Bombers",
        "2025-08-08"
    )
except ValueError as e:
    print("Test 1 passed:", e)


# Test 2: Unknown away team
try:
    predict_match_winner(
        "Geelong Cats",
        "Unknown FC",
        "2025-08-08"
    )
except ValueError as e:
    print("Test 2 passed:", e)


# Test 3: Invalid date
try:
    predict_match_winner(
        "Geelong Cats",
        "Essendon Bombers",
        "not-a-date"
    )
except ValueError as e:
    print("Test 3 passed:", e)


# Test 4: Date outside available dataset
try:
    predict_match_winner(
        "Geelong Cats",
        "Essendon Bombers",
        "2035-08-08"
    )
except ValueError as e:
    print("Test 4 passed:", e)


# Test 5: Same team entered twice
try:
    predict_match_winner(
        "Geelong Cats",
        "Geelong Cats",
        "2025-08-08"
    )
except ValueError as e:
    print("Test 5 passed:", e)

Test 1 passed: Unknown team name: 'Unknown FC'. Please use a team name from the dataset.
Test 2 passed: Unknown team name: 'Unknown FC'. Please use a team name from the dataset.
Test 3 passed: Invalid date. Please provide a valid date such as '2025-08-08'.
Test 4 passed: Date '2035-08-08' is outside the available data range (1983-03-26 to 2025-09-27).
Test 5 passed: team_a and team_b must be different teams.


### Predict Top Layer

In [23]:
def predict_top_player(match_date, team, top_k=5):

    # Validate team
    if not isinstance(team, str) or not team.strip():
        raise ValueError("Team name must be a non-empty string.")

    # Validate top_k
    if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    # Validate team exists
    known_teams = set(player_features["team"].dropna().unique())

    if team not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team}'. Please use a team name from the dataset."
        )

    # Validate date
    try:
        match_date = pd.to_datetime(match_date)
    except Exception:
        raise ValueError(
            "Invalid date. Please provide a valid date such as '2025-08-08'."
        )

    # Validate date range
    min_date = player_features["match_date"].min()
    max_date = player_features["match_date"].max()

    if match_date < min_date or match_date > max_date:
        raise ValueError(
            f"Date '{match_date.date()}' is outside the available data range "
            f"({min_date.date()} to {max_date.date()})."
        )

    # Get players for the requested team and date
    match_players = player_features[
        (player_features["match_date"] == match_date) &
        (player_features["team"] == team)
    ].copy()

    if match_players.empty:
        raise ValueError(
            f"No player data found for {team} on {match_date.date()}."
        )

    # Check whether usable features exist
    if match_players["player_recent_5_avg_disposals"].isna().all():
        raise ValueError(
            f"No usable player features found for {team} on {match_date.date()}."
        )

    # Predict disposals
    X_players = match_players[
        ["player_recent_5_avg_disposals"]
    ]

    match_players["predicted_disposals"] = (
        player_regression_pipeline.predict(X_players)
    )

    # Rank players
    ranked_players = (
        match_players
        .sort_values("predicted_disposals", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    # Build output
    results = []

    for rank, (_, row) in enumerate(ranked_players.iterrows(), start=1):
        results.append({
            "rank": rank,
            "player_id": int(row["player_id"]),
            "predicted_disposals": round(
                float(row["predicted_disposals"]), 2
            )
        })

    return results

In [105]:
top_players = predict_top_player(
    "2025-08-08",
    "Geelong Cats",
    top_k=5
)

for player in top_players:
    print(player)

{'rank': 1, 'player_id': 44960, 'predicted_disposals': 26.78}
{'rank': 2, 'player_id': 44073, 'predicted_disposals': 26.68}
{'rank': 3, 'player_id': 44487, 'predicted_disposals': 23.54}
{'rank': 4, 'player_id': 43312, 'predicted_disposals': 19.44}
{'rank': 5, 'player_id': 43717, 'predicted_disposals': 17.85}


Validation Test

In [106]:
# Test 1: Unknown team
try:
    predict_top_player(
        "2025-08-08",
        "Unknown FC",
        top_k=5
    )
except ValueError as e:
    print("Test 1 passed:", e)


# Test 2: Invalid date
try:
    predict_top_player(
        "not-a-date",
        "Geelong Cats",
        top_k=5
    )
except ValueError as e:
    print("Test 2 passed:", e)


# Test 3: Date outside available range
try:
    predict_top_player(
        "2035-08-08",
        "Geelong Cats",
        top_k=5
    )
except ValueError as e:
    print("Test 3 passed:", e)


# Test 4: Valid team/date but no player data
try:
    predict_top_player(
        "2025-08-08",
        "Geelong Cats",
        top_k=5
    )
    print("Test 4: valid request — player data found.")
except ValueError as e:
    print("Test 4:", e)

Test 1 passed: Unknown team name: 'Unknown FC'. Please use a team name from the dataset.
Test 2 passed: Invalid date. Please provide a valid date such as '2025-08-08'.
Test 3 passed: Date '2035-08-08' is outside the available data range (1983-03-27 to 2025-09-27).
Test 4: valid request — player data found.


In [26]:
# Test invalid top_k values

invalid_top_k_values = [0, -1, "5", None]

for value in invalid_top_k_values:
    try:
        predict_top_player(
            "2025-08-08",
            "Geelong Cats",
            top_k=value
        )
        print(f"FAILED: top_k={value} was accepted.")
    except (ValueError, TypeError) as e:
        print(f"Test passed for top_k={value}: {e}")

Test passed for top_k=0: top_k must be a positive integer.
Test passed for top_k=-1: top_k must be a positive integer.
Test passed for top_k=5: top_k must be a positive integer.
Test passed for top_k=None: top_k must be a positive integer.


In [107]:
# Final valid test for top-player prediction

result = predict_top_player(
    match_date="2025-08-08",
    team="Geelong Cats",
    top_k=5
)

print("Top-player prediction succeeded:")
for player in result:
    print(player)

Top-player prediction succeeded:
{'rank': 1, 'player_id': 44960, 'predicted_disposals': 26.78}
{'rank': 2, 'player_id': 44073, 'predicted_disposals': 26.68}
{'rank': 3, 'player_id': 44487, 'predicted_disposals': 23.54}
{'rank': 4, 'player_id': 43312, 'predicted_disposals': 19.44}
{'rank': 5, 'player_id': 43717, 'predicted_disposals': 17.85}


### Pridict.py

In [28]:
%%writefile predict.py

from pathlib import Path
import joblib
import pandas as pd


# File paths

BASE_DIR = Path(__file__).resolve().parent

MATCH_MODEL_PATH = BASE_DIR / "model_artifacts" / "match_winner_pipeline.joblib"
PLAYER_MODEL_PATH = BASE_DIR / "model_artifacts" / "top_player_pipeline.joblib"

MATCH_FEATURES_PATH = BASE_DIR / "afl_match_features_v1.csv"
PLAYER_FEATURES_PATH = BASE_DIR / "afl_player_features_v1.csv"


# Load trained models

match_winner_pipeline = joblib.load(MATCH_MODEL_PATH)
top_player_pipeline = joblib.load(PLAYER_MODEL_PATH)


# Load feature data

match_features = pd.read_csv(MATCH_FEATURES_PATH)
player_features = pd.read_csv(PLAYER_FEATURES_PATH)

match_features["match_date"] = pd.to_datetime(
    match_features["match_date"]
)

player_features["match_date"] = pd.to_datetime(
    player_features["match_date"]
)


# Match model feature columns

MATCH_FEATURE_COLUMNS = [
    "round",
    "home_team",
    "away_team",
    "venue",
    "home_recent_5_win_rate",
    "home_win_streak",
    "home_recent_5_avg_score",
    "home_days_rest",
    "away_recent_5_win_rate",
    "away_win_streak",
    "away_recent_5_avg_score",
    "away_days_rest",
    "h2h_matches",
    "h2h_current_home_wins",
    "h2h_current_away_wins",
    "h2h_draws",
    "h2h_current_home_win_rate",
    "home_pre_match_ladder_rank",
    "home_pre_match_points",
    "home_pre_match_percentage",
    "away_pre_match_ladder_rank",
    "away_pre_match_points",
    "away_pre_match_percentage"
]


# Predict match winner

def predict_match_winner(team_a, team_b, date):

    # Validate team names
    if not isinstance(team_a, str) or not team_a.strip():
        raise ValueError("team_a must be a non-empty string.")

    if not isinstance(team_b, str) or not team_b.strip():
        raise ValueError("team_b must be a non-empty string.")

    # Teams must be different
    if team_a == team_b:
        raise ValueError("team_a and team_b must be different teams.")

    # Validate date
    try:
        match_date = pd.to_datetime(date)
    except Exception:
        raise ValueError(
            "Invalid date. Please provide a valid date such as '2025-08-08'."
        )

    # Known teams
    known_teams = set(
        match_features["home_team"].dropna().unique()
    ) | set(
        match_features["away_team"].dropna().unique()
    )

    if team_a not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team_a}'. "
            "Please use a team name from the dataset."
        )

    if team_b not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team_b}'. "
            "Please use a team name from the dataset."
        )

    # Available date range
    min_date = match_features["match_date"].min()
    max_date = match_features["match_date"].max()

    if match_date < min_date or match_date > max_date:
        raise ValueError(
            f"Date '{match_date.date()}' is outside the available "
            f"data range ({min_date.date()} to {max_date.date()})."
        )

    # Find exact match
    match_row = match_features[
        (match_features["match_date"] == match_date) &
        (match_features["home_team"] == team_a) &
        (match_features["away_team"] == team_b)
    ].copy()

    if match_row.empty:
        raise ValueError(
            f"No match found for {team_a} vs {team_b} "
            f"on {match_date.date()}."
        )

    # Prepare model input
    X = match_row[MATCH_FEATURE_COLUMNS]

    # Prediction
    prediction = match_winner_pipeline.predict(X)[0]
    probabilities = match_winner_pipeline.predict_proba(X)[0]

    class_names = match_winner_pipeline.classes_

    probability_map = {
        class_name: float(probability)
        for class_name, probability
        in zip(class_names, probabilities)
    }

    return {
        "home_team": team_a,
        "away_team": team_b,
        "date": str(match_date.date()),
        "winner": prediction,
        "probability": round(
            probability_map[prediction],
            4
        ),
        "class_probabilities": {
            key: round(value, 4)
            for key, value in probability_map.items()
        }
    }


# Predict top players

def predict_top_player(match_date, team, top_k=5):

    # Validate team
    if not isinstance(team, str) or not team.strip():
        raise ValueError(
            "Team name must be a non-empty string."
        )

    # Validate top_k
    if (
        not isinstance(top_k, int)
        or isinstance(top_k, bool)
        or top_k <= 0
    ):
        raise ValueError(
            "top_k must be a positive integer."
        )

    # Validate team exists
    known_teams = set(
        player_features["team"].dropna().unique()
    )

    if team not in known_teams:
        raise ValueError(
            f"Unknown team name: '{team}'. "
            "Please use a team name from the dataset."
        )

    # Validate date
    try:
        match_date = pd.to_datetime(match_date)
    except Exception:
        raise ValueError(
            "Invalid date. Please provide a valid date such as '2025-08-08'."
        )

    # Available date range
    min_date = player_features["match_date"].min()
    max_date = player_features["match_date"].max()

    if match_date < min_date or match_date > max_date:
        raise ValueError(
            f"Date '{match_date.date()}' is outside the available "
            f"data range ({min_date.date()} to {max_date.date()})."
        )

    # Get players for requested team and date
    match_players = player_features[
        (player_features["match_date"] == match_date) &
        (player_features["team"] == team)
    ].copy()

    if match_players.empty:
        raise ValueError(
            f"No player data found for {team} "
            f"on {match_date.date()}."
        )

    # Check for usable features
    if match_players[
        "player_recent_5_avg_disposals"
    ].isna().all():

        raise ValueError(
            f"No usable player features found for {team} "
            f"on {match_date.date()}."
        )

    # Model input
    X_players = match_players[
        ["player_recent_5_avg_disposals"]
    ]

    # Predict
    match_players["predicted_disposals"] = (
        top_player_pipeline.predict(X_players)
    )

    # Rank
    ranked_players = (
        match_players
        .sort_values(
            "predicted_disposals",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    # Build result
    results = []

    for rank, (_, row) in enumerate(
        ranked_players.iterrows(),
        start=1
    ):

        results.append({
            "rank": rank,
            "player_id": int(row["player_id"]),
            "predicted_disposals": round(
                float(row["predicted_disposals"]),
                2
            )
        })

    return results

Overwriting predict.py


In [108]:
# Verify that predict.py works independently

import sys
import importlib

# Reload if it was already imported
if "predict" in sys.modules:
    del sys.modules["predict"]

import predict

print("predict.py imported successfully.")
print("Match model loaded:", predict.match_winner_pipeline is not None)
print("Top-player model loaded:", predict.top_player_pipeline is not None)
print("Match features loaded:", predict.match_features.shape)
print("Player features loaded:", predict.player_features.shape)

predict.py imported successfully.
Match model loaded: True
Top-player model loaded: True
Match features loaded: (7904, 26)
Player features loaded: (274089, 7)


In [109]:
# Test the functions directly from predict.py

# Test 1: Match winner prediction

match_result = predict.predict_match_winner(
    "Geelong Cats",
    "Essendon Bombers",
    "2025-08-08"
)

print("Match Winner Prediction:")
print(match_result)


# Test 2: Top-player prediction

player_result = predict.predict_top_player(
    "2025-08-08",
    "Geelong Cats",
    top_k=5
)

print("\nTop Player Prediction:")

for player in player_result:
    print(player)

Match Winner Prediction:
{'home_team': 'Geelong Cats', 'away_team': 'Essendon Bombers', 'date': '2025-08-08', 'winner': 'Home Win', 'probability': 0.9165, 'class_probabilities': {'Away Win': 0.0795, 'Draw': 0.004, 'Home Win': 0.9165}}

Top Player Prediction:
{'rank': 1, 'player_id': 44960, 'predicted_disposals': 26.78}
{'rank': 2, 'player_id': 44073, 'predicted_disposals': 26.68}
{'rank': 3, 'player_id': 44487, 'predicted_disposals': 23.54}
{'rank': 4, 'player_id': 43312, 'predicted_disposals': 19.44}
{'rank': 5, 'player_id': 43717, 'predicted_disposals': 17.85}


In [110]:
# Final validation tests using the standalone predict.py functions

tests = [
    (
        "Unknown team",
        lambda: predict.predict_match_winner(
            "Unknown FC",
            "Geelong Cats",
            "2025-08-08"
        )
    ),
    (
        "Invalid date",
        lambda: predict.predict_match_winner(
            "not-a-date",
            "Geelong Cats",
            "2025-08-08"
        )
    ),
    (
        "Out-of-range date",
        lambda: predict.predict_match_winner(
            "Geelong Cats",
            "Essendon Bombers",
            "2035-08-08"
        )
    ),
    (
        "Same teams",
        lambda: predict.predict_match_winner(
            "Geelong Cats",
            "Geelong Cats",
            "2025-08-08"
        )
    ),
    (
        "Invalid top_k",
        lambda: predict.predict_top_player(
            "2025-08-08",
            "Geelong Cats",
            top_k=0
        )
    ),
    (
        "Unknown player team",
        lambda: predict.predict_top_player(
            "2025-08-08",
            "Unknown FC",
            top_k=5
        )
    )
]

for test_name, test_function in tests:
    try:
        test_function()
        print(f"FAILED: {test_name}")
    except (ValueError, TypeError) as e:
        print(f"PASSED: {test_name} -> {e}")

PASSED: Unknown team -> Unknown team name: 'Unknown FC'. Please use a team name from the dataset.
PASSED: Invalid date -> Unknown team name: 'not-a-date'. Please use a team name from the dataset.
PASSED: Out-of-range date -> Date '2035-08-08' is outside the available data range (1983-03-26 to 2025-09-27).
PASSED: Same teams -> team_a and team_b must be different teams.
PASSED: Invalid top_k -> top_k must be a positive integer.
PASSED: Unknown player team -> Unknown team name: 'Unknown FC'. Please use a team name from the dataset.


In [111]:
# Correct invalid-date validation test

try:
    predict.predict_match_winner(
        "Geelong Cats",
        "Essendon Bombers",
        "not-a-date"
    )
    print("FAILED: Invalid date was accepted.")
except ValueError as e:
    print("PASSED: Invalid date ->", e)

PASSED: Invalid date -> Invalid date. Please provide a valid date such as '2025-08-08'.


###  How to Call the Packaged Models

#### 1. Match Winner Prediction

The packaged match model can be called using:

```python
from predict import predict_match_winner

result = predict_match_winner(
    team_a="Geelong Cats",
    team_b="Essendon Bombers",
    date="2025-08-08"
)

print(result)

Example output:
{
    "home_team": "Geelong Cats",
    "away_team": "Essendon Bombers",
    "date": "2025-08-08",
    "winner": "Home Win",
    "probability": 0.9165,
    "class_probabilities": {
        "Away Win": 0.0795,
        "Draw": 0.004,
        "Home Win": 0.9165
    }
}

####2. Top Player Prediction
The packaged top-player model can be called using:

from predict import predict_top_player

result = predict_top_player(
    match_date="2025-08-08",
    team="Geelong Cats",
    top_k=5
)

for player in result:
    print(player)


Example output:
[
    {"rank": 1, "player_id": 44960, "predicted_disposals": 26.78},
    {"rank": 2, "player_id": 44073, "predicted_disposals": 26.68},
    {"rank": 3, "player_id": 44487, "predicted_disposals": 23.54},
    {"rank": 4, "player_id": 43312, "predicted_disposals": 19.44},
    {"rank": 5, "player_id": 43717, "predicted_disposals": 17.85}
]